# Sprout (Qwen2.5-1.5B SFT, merged) on vLLM, one Colab T4

Same load generator and prompts as the MacBook runs (`scripts/bench_openai.py`, agent prompts with retrieved facts),
so the numbers line up with `results/serving/qwen_*`:

| run | what it answers |
|---|---|
| closed loop, fixed 160 tokens (`--ignore-eos`), 1 to 64 users | capacity: throughput and TPOT as users grow |
| closed loop, natural lengths, 1 to 64 users | what the agent actually sees; SLO share |
| open loop, Poisson 1 to 32 req/s | goodput: requests/s served within the SLOs (TTFT 1 s, TPOT 0.1 s, E2E 10 s) |

Results go to Drive (`llm_eval_runs/serving_t4/results/serving`); a finished run is skipped when the notebook is run
again, so a disconnect loses at most one run. Prefix caching was measured before (old notebook,
`serving/vllm_serving_benchmark.ipynb`); the agent prompts here share only the short system prompt, so vLLM's default
(caching on) is kept.

In [ ]:
!pip install -q -U vllm openai peft   # once per runtime

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os, sys, json, time, subprocess, shutil, urllib.request
D = '/content/drive/MyDrive/llm_eval_runs'
CODE = f'{D}/code'; RUN = f'{D}/serving_t4'
BASE = 'Qwen/Qwen2.5-1.5B-Instruct'
ADAPTER = '/content/drive/MyDrive/llm_sft_runs/sft_lora_qwen_rag/E_alpha64/adapter'
MERGED = f'{D}/sprout-qwen-merged'                   # shared with eval/safety_e2e_colab.ipynb
os.makedirs(f'{RUN}/results/serving', exist_ok=True); os.makedirs(f'{RUN}/logs', exist_ok=True)
os.chdir(RUN)                                        # bench reads results/*.jsonl prompts, writes results/serving/
GPU = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print(GPU)
if not os.path.exists(f'{MERGED}/config.json'):       # merge once: fp32 on CPU, saved fp16 (= scripts/merge_lora.py)
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    m = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float32)
    m = PeftModel.from_pretrained(m, ADAPTER).merge_and_unload()
    m.to(torch.float16).save_pretrained(MERGED, safe_serialization=True)
    AutoTokenizer.from_pretrained(BASE).save_pretrained(MERGED)
    del m; import gc; gc.collect(); print('merged ->', MERGED)
LOCAL = '/content/sprout-qwen'                        # load from local disk, not the Drive mount
if not os.path.exists(LOCAL): shutil.copytree(MERGED, LOCAL)

In [ ]:
VLLM = os.path.join(os.path.dirname(sys.executable), 'vllm')
VLLM = VLLM if os.path.exists(VLLM) else shutil.which('vllm')
def start(tag, *flags):
    log = open(f'{RUN}/logs/server_{tag}.log', 'w')
    p = subprocess.Popen([VLLM, 'serve', LOCAL, '--served-model-name', 'sprout', '--dtype', 'half',
                          '--max-model-len', '2048', '--gpu-memory-utilization', '0.90', '--port', '8000', *flags],
                         stdout=log, stderr=subprocess.STDOUT)
    for _ in range(120):
        try:
            urllib.request.urlopen('http://localhost:8000/v1/models', timeout=2); print('server up'); return p
        except Exception:
            if p.poll() is not None: raise RuntimeError(open(log.name).read()[-3000:])
            time.sleep(5)
    raise RuntimeError('server did not start; see ' + log.name)
def stop(p):
    p.terminate(); p.wait(60); time.sleep(5)
def bench(tag, *args):
    if os.path.exists(f'results/serving/{tag}.json'):
        print(tag, 'done'); return
    t0 = time.time()
    subprocess.run([sys.executable, f'{CODE}/bench_openai.py', '--base-url', 'http://localhost:8000/v1',
                    '--model', 'sprout', '--tag', tag, *args], check=True)
    print(tag, f'{time.time() - t0:.0f} s')

In [ ]:
server = start('default')
bench('vllm_t4_warmup', '--concurrency', '4', '--requests', '8')
# fixed lengths: every answer runs to 160 tokens, the same work per request as the Mac `_fixed` runs
bench('vllm_t4_fixed', '--ignore-eos', '--concurrency', '1,2,4,8', '--requests', '32')
bench('vllm_t4_fixed_hi', '--ignore-eos', '--concurrency', '16,32,64', '--requests', '128')
# natural lengths (the model's own stop), as the agent runs
bench('vllm_t4_stop', '--concurrency', '1,2,4,8', '--requests', '32')
bench('vllm_t4_stop_hi', '--concurrency', '16,32,64', '--requests', '128')
# open loop: Poisson arrivals, goodput under the SLOs
bench('vllm_t4_open', '--mode', 'open', '--rates', '1,2,4,8,16,32', '--requests', '128')
stop(server)
json.dump(dict(gpu=GPU, vllm=subprocess.run([sys.executable, '-c', 'import vllm; print(vllm.__version__)'],
                                            capture_output=True, text=True).stdout.strip()),
          open('results/serving/vllm_t4_env.json', 'w'))

In [ ]:
!python {CODE}/analyze_serving.py --runs vllm_t4_fixed,vllm_t4_fixed_hi,vllm_t4_stop,vllm_t4_stop_hi --dir results/serving --md results/serving/vllm_t4.md
!python {CODE}/analyze_serving.py --runs vllm_t4_open --dir results/serving --md results/serving/vllm_t4_open.md